# Simple Backtest

Walks each historical DFS snapshot in `dfs_data/`, runs the same flow as `live_notebook.ipynb`:

`predict_min_times_rate` → `line_probs_for_market` → `build_greedy_slate` (top 2 pairs)

Then grades the slate against actual box-score results.

In [2]:
import pandas as pd
import numpy as np
import warnings
import joblib
import json
import re
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import nameDict
from src.pipeline.props_pipeline.ppm_pipeline import ppm_pipeline
from src.pipeline.props_pipeline.apm_pipeline import apm_pipeline
from src.pipeline.props_pipeline.min_pipeline import min_pipeline
from src.live import (
    predict_min_times_rate,
    run_pts_simulation,
    line_probs_for_market,
    build_greedy_slate,
)

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
np.random.seed(42)

print("Setup complete.")

Setup complete.


### Load training data and saved models

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26], ignore_index=True)
base_df['GAME_DATE'] = pd.to_datetime(base_df['GAME_DATE'])

pts_df_full = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df_full = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
min_df_full = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')
for _df in (pts_df_full, ast_df_full, min_df_full):
    _df['GAME_DATE'] = pd.to_datetime(_df['GAME_DATE'])

min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]

ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]

apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]

print(f"base_df: {len(base_df):,} rows   |   date range: {base_df['GAME_DATE'].min().date()} → {base_df['GAME_DATE'].max().date()}")

base_df: 52,986 rows   |   date range: 2024-10-22 → 2026-04-12


### Helpers

`build_greedy_slate` expects a handful of context columns from the live notebook's `generalized_best_bets` enrichment step. For a simple backtest we don't have the US team lines for historical dates, so we plug in safe defaults (odds come directly from the DFS snapshot).

In [4]:
STAT_COL_BY_MARKET = {"PTS": "PTS", "AST": "AST", "REB": "REB"}

def american_to_prob(odds):
    try:
        o = float(odds)
    except (TypeError, ValueError):
        return np.nan
    if np.isnan(o):
        return np.nan
    return abs(o) / (abs(o) + 100.0) if o < 0 else 100.0 / (o + 100.0)


def add_slate_defaults(line_probs, lines_df):
    """Attach odds from the DFS snapshot and stub the rest of `build_greedy_slate`'s columns."""
    df = line_probs.copy()
    if df.empty:
        return df

    odds_piv = (
        lines_df.pivot_table(index=['NAME', 'LINE'], columns='OVER/UNDER', values='ODDS', aggfunc='first')
                .reset_index()
    )
    odds_piv = odds_piv.rename(columns={'Over': 'ODDS_OVER', 'Under': 'ODDS_UNDER'})
    for col in ('ODDS_OVER', 'ODDS_UNDER'):
        if col not in odds_piv.columns:
            odds_piv[col] = np.nan

    df = df.merge(
        odds_piv[['NAME', 'LINE', 'ODDS_OVER', 'ODDS_UNDER']],
        left_on=['PLAYER_NAME', 'LINE'], right_on=['NAME', 'LINE'], how='left',
    ).drop(columns=['NAME'])
    df['ODDS_OVER']  = df['ODDS_OVER'].fillna(-137)
    df['ODDS_UNDER'] = df['ODDS_UNDER'].fillna(-137)
    df['IMP_PROB_OVER']  = df['ODDS_OVER'].apply(american_to_prob)
    df['IMP_PROB_UNDER'] = df['ODDS_UNDER'].apply(american_to_prob)

    df['OPPONENT'] = ''
    numeric_stub_cols = [
        'TEAM_SPREAD', 'GAME_TOTAL', 'OPP_DEF_RATING', 'OPP_RANK_DEF_RATING',
        'OPP_PACE', 'OPP_PACE_RANK',
        'EDGE', 'MED_EDGE', 'Z_SCORE', 'EV_OVER', 'EV_UNDER',
        'AVG_STAT_L10', 'MED_STAT_L10', 'STD_STAT_L10',
        'OVER_RATE_L5', 'OVER_RATE_L10', 'OVER_RATE_L15', 'OVER_RATE_SEASON',
        'AVG_MIN_L10', 'STD_MIN_L10', 'AVG_USG_L10', 'STD_USG_L10',
        'AVG_STAT_VS_MATCHUP',
    ]
    for c in numeric_stub_cols:
        df[c] = 0.0
    df['MATCHUP_GAMES'] = 0
    return df


def grade_slate(slate_rows, actuals_by_name):
    """Score a 2-leg slate. Each leg hits if the actual value beats/misses the line per SIDE."""
    graded = []
    for row in slate_rows:
        legs = []
        for i in (1, 2):
            name, market, line, side = row[f'NAME {i}'], row[f'MARKET {i}'], row[f'LINE {i}'], row[f'SIDE {i}']
            stat_col = STAT_COL_BY_MARKET.get(market)
            actual = actuals_by_name.get((name, stat_col)) if stat_col else None
            if actual is None or (isinstance(actual, float) and np.isnan(actual)):
                legs.append({'name': name, 'market': market, 'line': line, 'side': side,
                             'actual': None, 'hit': None})
                continue
            hit = (actual > line) if side == 'over' else (actual < line)
            legs.append({'name': name, 'market': market, 'line': line, 'side': side,
                         'actual': float(actual), 'hit': bool(hit)})
        missing = any(l['hit'] is None for l in legs)
        parlay_hit = None if missing else all(l['hit'] for l in legs)
        graded.append({
            'pair': f"{row['NAME 1']} ({row['MARKET 1']} {row['SIDE 1']} {row['LINE 1']}) + "
                    f"{row['NAME 2']} ({row['MARKET 2']} {row['SIDE 2']} {row['LINE 2']})",
            'parlay_prob': row.get('PARLAY_PROB'),
            'ev': row.get('EV'),
            'legs': legs,
            'parlay_hit': parlay_hit,
        })
    return graded


DATE_RE = re.compile(r'NBA_DFS_(\d{8})')

def collect_dfs_files(folder='dfs_data'):
    """One file per slate date (the lexicographically last snapshot of the day)."""
    by_date = {}
    for f in sorted(Path(folder).glob('NBA_DFS_*.csv')):
        m = DATE_RE.match(f.name)
        if not m:
            continue
        date = datetime.strptime(m.group(1), '%Y%m%d').date()
        by_date[date] = f
    return sorted(by_date.items())

dfs_files = collect_dfs_files()
print(f"Found {len(dfs_files)} slate dates ({dfs_files[0][0]} → {dfs_files[-1][0]}).")

Found 56 slate dates (2025-10-10 → 2025-12-14).


### Run backtest

For each slate date we:
1. Filter training + base data to games strictly before that date.
2. `predict_min_times_rate` for PTS and AST.
3. `line_probs_for_market` per market, concat.
4. Attach DFS odds + stub context fields, then `build_greedy_slate(top_n=2)`.
5. Grade the 2 pairs against the actual box score on that date.

In [5]:
BOOKMAKER = 'PrizePicks'   # flip to 'Underdog', 'Betr DFS', etc. as needed
TOP_N     = 2
OUTPUT_DIR = Path('data/props/backtest')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results = []

for slate_date, csv_path in dfs_files:
    current_date = slate_date.strftime('%Y-%m-%d')
    cutoff = pd.Timestamp(slate_date)

    pts_df = pts_df_full[pts_df_full['GAME_DATE'] < cutoff]
    ast_df = ast_df_full[ast_df_full['GAME_DATE'] < cutoff]
    min_df = min_df_full[min_df_full['GAME_DATE'] < cutoff]
    if min_df.empty or pts_df.empty:
        print(f"[SKIP] {current_date}: no training rows before date")
        continue

    lines_all = pd.read_csv(csv_path)
    lines_book = lines_all[lines_all['BOOKMAKER'] == BOOKMAKER]
    if lines_book.empty:
        print(f"[SKIP] {current_date}: no {BOOKMAKER} rows in {csv_path.name}")
        continue

    lines_pts = lines_book[lines_book['CATEGORY'] == 'player_points']
    lines_ast = lines_book[lines_book['CATEGORY'] == 'player_assists']
    pts_names = lines_pts['NAME'].unique()
    ast_names = lines_ast['NAME'].unique()

    pts_preds = predict_min_times_rate(
        pts_names, min_df, pts_df, current_date,
        name_dict=nameDict,
        rate_pipeline=ppm_pipeline,
        rate_quantile_models=ppm_quantile_models,
        min_quantile_models=min_quantile_models,
        stat_prefix="PTS",
        verbose=False,
    )
    ast_preds = predict_min_times_rate(
        ast_names, min_df, ast_df, current_date,
        name_dict=nameDict,
        rate_pipeline=apm_pipeline,
        rate_quantile_models=apm_quantile_models,
        min_quantile_models=min_quantile_models,
        stat_prefix="AST",
        verbose=False,
    )

    probs_pts = line_probs_for_market(pts_preds, lines_pts, nameDict, run_pts_simulation)
    probs_ast = line_probs_for_market(ast_preds, lines_ast, nameDict, run_pts_simulation)
    all_line_probs = pd.concat([probs_pts, probs_ast], ignore_index=True).dropna(subset=['LINE'])
    if all_line_probs.empty:
        print(f"[SKIP] {current_date}: no usable legs")
        continue

    enriched = add_slate_defaults(all_line_probs, lines_book)

    slate_path = build_greedy_slate(
        prob_df=enriched,
        min_df=min_df,
        min_ev=0.0,
        min_kelly=0.0,
        kelly_fraction=0.5,
        top_n=TOP_N,
        json_path=OUTPUT_DIR / f"{BOOKMAKER.replace(' ', '_')}_{slate_date.isoformat()}.json",
    )

    slate_rows = json.loads(Path(slate_path).read_text())

    actuals = base_df[base_df['GAME_DATE'].dt.date == slate_date]
    actuals_by_name = {}
    for _, r in actuals.iterrows():
        for stat in ('PTS', 'AST', 'REB'):
            actuals_by_name[(r['PLAYER_NAME'], stat)] = r[stat]

    graded = grade_slate(slate_rows, actuals_by_name)
    for g in graded:
        g['date'] = current_date
    results.extend(graded)

    hits = sum(1 for g in graded if g['parlay_hit'] is True)
    misses = sum(1 for g in graded if g['parlay_hit'] is False)
    print(f"{current_date} | pairs={len(graded)} | parlays W-L = {hits}-{misses}")

print(f"\nDone. Graded {len(results)} pairs across {len(dfs_files)} dates.")

No game found for HOU within 3 days from 2025-10-10
No game found for HOU within 3 days from 2025-10-10
Legs: 12  |  Pairs: 44  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/backtest/PrizePicks_2025-10-10.json
2025-10-10 | pairs=2 | parlays W-L = 0-0
No game found for PHX within 3 days from 2025-10-15
No game found for CLE within 3 days from 2025-10-15
No game found for CLE within 3 days from 2025-10-15
No game found for CLE within 3 days from 2025-10-15
No game found for CLE within 3 days from 2025-10-15
No game found for PHX within 3 days from 2025-10-15
Legs: 28  |  Pairs: 243  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/backtest/PrizePicks_2025-10-15.json
2025-10-15 | pairs=2 | parlays W-L = 0-0
No game found for PHX within 3 days from 2025-10-16
No game found for CLE within 3 days from 2025-10-16
No game found for CLE within 3 days from 2025-10-16
No game found for CLE within 3 days from 2025-10-16
No game f

KeyError: 'BOOKMAKER'

### Summary

In [ ]:
rows = []
for r in results:
    for i, leg in enumerate(r['legs'], start=1):
        rows.append({
            'date':    r['date'],
            'pair':    r['pair'],
            'leg':     i,
            'player':  leg['name'],
            'market':  leg['market'],
            'line':    leg['line'],
            'side':    leg['side'],
            'actual':  leg['actual'],
            'leg_hit': leg['hit'],
            'parlay_hit':  r['parlay_hit'],
            'parlay_prob': r['parlay_prob'],
            'ev':          r['ev'],
        })
legs_df = pd.DataFrame(rows)

pairs_df = pd.DataFrame([
    {'date': r['date'], 'pair': r['pair'], 'parlay_hit': r['parlay_hit'],
     'parlay_prob': r['parlay_prob'], 'ev': r['ev']}
    for r in results
])

leg_graded = legs_df[legs_df['leg_hit'].notna()]
pair_graded = pairs_df[pairs_df['parlay_hit'].notna()]

print("=== Backtest summary ===")
print(f"Dates run:           {pairs_df['date'].nunique()}")
print(f"Pairs generated:     {len(pairs_df)}")
print(f"Pairs graded:        {len(pair_graded)}")
if len(pair_graded):
    hit_rate = pair_graded['parlay_hit'].mean()
    print(f"Parlay hit rate:     {hit_rate:.1%}  ({int(pair_graded['parlay_hit'].sum())}/{len(pair_graded)})")
    pnl = pair_graded['parlay_hit'].map({True: 2.0, False: -1.0}).sum()
    print(f"Units (+2/-1):       {pnl:+.1f}  across {len(pair_graded)} pairs  |  ROI {pnl/len(pair_graded):+.1%}")
if len(leg_graded):
    print(f"Leg hit rate:        {leg_graded['leg_hit'].mean():.1%}  ({int(leg_graded['leg_hit'].sum())}/{len(leg_graded)})")

legs_df.head(20)

In [ ]:
if len(pair_graded):
    by_date = (
        pair_graded.assign(win=pair_graded['parlay_hit'].astype(int))
                   .groupby('date')
                   .agg(pairs=('win', 'size'), wins=('win', 'sum'))
                   .assign(hit_rate=lambda d: d['wins'] / d['pairs'])
                   .reset_index()
    )
    display(by_date)